# Сравнение векторов документа Current с Up и Down используя ChatGPT.

Сравнивает вектора документа с метаданными `Current` c векторами документов с метаданными `Up` и `Down`, и определяет к какой категории документ `Current` ближе в процентном отношении.

In [1]:
from dotenv import load_dotenv
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
import os
import yaml
from pathlib import Path
import chromadb
import shutil
import hashlib
import numpy as np
import logging
from openai import OpenAIError

# Настройка логирования
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Пути и параметры
md_path = Path('c:/Users/Alkor/gd/news_rss_md')
chromadb_path = './chroma_db_chatgpt_compare'
# model_name = "text-embedding-3-small"  # Модель OpenAI для эмбеддингов
model_name = "text-embedding-3-large"  # Модель OpenAI для эмбеддингов
batch_size = 5  # Размер батча для добавления документов

# Загрузка ключа API ChatGPT из .env файла
load_dotenv(override=True)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

# Кастомная функция эмбеддингов для ChromaDB
class OpenAIEmbeddingFunction:
    def __init__(self, api_key, model_name):
        self.embeddings = OpenAIEmbeddings(api_key=api_key, model=model_name)
    
    def __call__(self, input):
        return self.embeddings.embed_documents(input)

def get_folder_size(folder_path):
    total_size = 0
    for dirpath, _, filenames in os.walk(folder_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)  # Размер в МБ

def load_markdown_files(directory):
    documents = []
    for file_path in list(directory.glob("**/*.md")):
        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                content = file.read()
            
            # Разделение метаданных и текста
            if content.startswith('---'):
                parts = content.split('---', 2)
                if len(parts) >= 3:
                    metadata_yaml = parts[1].strip()
                    text_content = parts[2].strip()
                    metadata = yaml.safe_load(metadata_yaml)
                    doc = Document(
                        page_content=text_content,
                        metadata={
                            "next_bar": metadata.get("next_bar", ""),
                            "source": file_path.name,
                            "date": file_path.stem
                        }
                    )
                    documents.append(doc)
                else:
                    doc = Document(
                        page_content=content,
                        metadata={
                            "next_bar": "unknown",
                            "source": file_path.name,
                            "date": file_path.stem
                        }
                    )
                    documents.append(doc)
            else:
                doc = Document(
                    page_content=content,
                    metadata={
                        "next_bar": "unknown",
                        "source": file_path.name,
                        "date": file_path.stem
                    }
                )
                documents.append(doc)
        except Exception as e:
            logger.error(f"Ошибка при обработке файла {file_path}: {e}")
    return documents

# Косинусное сходство
def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot_product / (norm1 * norm2)

# Функция для добавления документов батчами
def add_in_batches(collection, ids, documents, metadatas, batch_size):
    for i in range(0, len(documents), batch_size):
        batch_ids = ids[i:i + batch_size]
        batch_docs = documents[i:i + batch_size]
        batch_metas = metadatas[i:i + batch_size]
        try:
            logger.info(f"Добавление батча {i // batch_size + 1} с {len(batch_docs)} документами")
            collection.add(ids=batch_ids, documents=batch_docs, metadatas=batch_metas)
            logger.info(f"Батч {i // batch_size + 1} успешно добавлен")
        except OpenAIError as e:
            logger.error(f"Ошибка OpenAI API при добавлении батча {i // batch_size + 1}: {e}")
            raise
        except Exception as e:
            logger.error(f"Общая ошибка при добавлении батча {i // batch_size + 1}: {e}")
            raise

# Удаление папки chroma_db, если она существует
if os.path.exists(chromadb_path):
    print(f"Размер папки {chromadb_path} до удаления: {get_folder_size(chromadb_path):.2f} МБ")
    shutil.rmtree(chromadb_path)
    print(f"Папка {chromadb_path} удалена.")

# Инициализация клиента ChromaDB
client = chromadb.PersistentClient(path=chromadb_path)

# Создание функции эмбеддингов для OpenAI
ef = OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name=model_name
)

# Создание коллекции
collection = client.create_collection(name="news_collection", embedding_function=ef)

# Загрузка Markdown-файлов
documents = load_markdown_files(md_path)

# Проверка на пустую папку
if not documents:
    print("Не найдено Markdown-файлов в указанной директории.")
    exit(1)
else:
    print(f"Загружено {len(documents)} Markdown-файлов из {md_path}")
    print(f"Документы даты: {set(doc.metadata['date'] for doc in documents)}")
    print(f"Направление следующего бара: {set(doc.metadata['next_bar'] for doc in documents)}")

# Подготовка данных для ChromaDB
doc_texts = [doc.page_content for doc in documents]
doc_ids = [hashlib.md5(doc.page_content.encode()).hexdigest() for doc in documents]
doc_metadatas = [doc.metadata for doc in documents]

# Добавление в коллекцию батчами
try:
    add_in_batches(collection, doc_ids, doc_texts, doc_metadatas, batch_size)
    logger.info("Все документы успешно добавлены.")
except Exception as e:
    logger.error(f"Ошибка при добавлении документов: {e}")
    exit(1)

# Сравнение векторов
def compare_vectors():
    # Получение документов с next_bar="current"
    current_results = collection.query(
        query_texts=[""],  # Пустой запрос, используем фильтр
        n_results=1000,  # Достаточно большой лимит
        where={"next_bar": "current"}
    )
    
    if not current_results['ids'][0]:
        print("Документы с next_bar='current' не найдены.")
        return
    
    current_id = current_results['ids'][0][0]
    current_embedding = collection.get(ids=[current_id], include=["embeddings"])["embeddings"][0]
    
    # Получение документов с next_bar="up"
    up_results = collection.query(
        query_texts=[""],
        n_results=1000,
        where={"next_bar": "up"}
    )
    
    # Получение документов с next_bar="down"
    down_results = collection.query(
        query_texts=[""],
        n_results=1000,
        where={"next_bar": "down"}
    )
    
    # Проверка наличия документов
    if not up_results['ids'][0]:
        print("Документы с next_bar='up' не найдены.")
        return
    if not down_results['ids'][0]:
        print("Документы с next_bar='down' не найдены.")
        return
    
    # Вычисление среднего сходства для up
    up_embeddings = collection.get(ids=up_results['ids'][0], include=["embeddings"])["embeddings"]
    up_similarities = [cosine_similarity(current_embedding, emb) for emb in up_embeddings]
    avg_up_similarity = np.mean(up_similarities) * 100  # В процентах
    
    # Вычисление среднего сходства для down
    down_embeddings = collection.get(ids=down_results['ids'][0], include=["embeddings"])["embeddings"]
    down_similarities = [cosine_similarity(current_embedding, emb) for emb in down_embeddings]
    avg_down_similarity = np.mean(down_similarities) * 100  # В процентах
    
    # Вывод результатов
    print(f"Среднее сходство с категорией 'up': {avg_up_similarity:.2f}%")
    print(f"Среднее сходство с категорией 'down': {avg_down_similarity:.2f}%")
    
    # Определение ближайшей категории
    if avg_up_similarity > avg_down_similarity:
        print("Документ с next_bar='current' ближе к категории 'up'.")
    elif avg_down_similarity > avg_up_similarity:
        print("Документ с next_bar='current' ближе к категории 'down'.")
    else:
        print("Документ с next_bar='current' имеет одинаковое сходство с категориями 'up' и 'down'.")

# Выполнение сравнения
compare_vectors()

# # Пример поиска с фильтрацией по метаданным
# query = "Новости о Tesla"
# results = collection.query(
#     query_texts=[query],
#     n_results=3,
#     where={"next_bar": "up"}
# )
# print("\nРезультаты поиска по запросу 'Новости о Tesla':")
# print(results)

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


Размер папки ./chroma_db_chatgpt_compare до удаления: 71.12 МБ
Папка ./chroma_db_chatgpt_compare удалена.


INFO:__main__:Добавление батча 1 с 5 документами


Загружено 21 Markdown-файлов из c:\Users\Alkor\gd\news_rss_md
Документы даты: {'current', '2025-06-30', '2025-07-08', '2025-06-26', '2025-07-22', '2025-06-25', '2025-07-11', '2025-07-01', '2025-07-16', '2025-07-03', '2025-07-10', '2025-07-21', '2025-07-15', '2025-07-18', '2025-07-17', '2025-06-27', '2025-07-04', '2025-07-09', '2025-07-07', '2025-07-02', '2025-07-14'}
Направление следующего бара: {'current', 'up', 'down'}


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:__main__:Батч 1 успешно добавлен
INFO:__main__:Добавление батча 2 с 5 документами
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:__main__:Батч 2 успешно добавлен
INFO:__main__:Добавление батча 3 с 5 документами
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:__main__:Батч 3 успешно добавлен
INFO:__main__:Добавление батча 4 с 5 документами
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:__main__:Батч 4 успешно добавлен
INFO:__main__:Добавление батча 5 с 1 документами
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:__main__:Батч 5 успешно добавлен
INFO:__main__:Все документы успешно добавлены.
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/e

Среднее сходство с категорией 'up': 84.56%
Среднее сходство с категорией 'down': 91.36%
Документ с next_bar='current' ближе к категории 'down'.
